# IQM Metric Test (CV Classification on Hardware)

Pilot → protocol selection → final run → paired statistical analysis for **odra** vs **simulator** ansatze on IQM Spark.

Reusable logic lives in `src/qbanknote` (`evaluation`, `stats`, `classification`, `weights`, `iqm`).

## 1. Imports & Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qbanknote.paths import ensure_importable, find_project_root

ensure_importable()

from qbanknote.classification import evaluate_predictions, predictions_to_labels
from qbanknote.evaluation import (
    build_run_dir,
    load_phase_spec,
    read_csv_or_empty,
    run_cv_experiment,
    summarize_results,
    timestamp_run_id,
)
from qbanknote.iqm import connect_to_iqm_backend
from qbanknote.stats import analyze_final_summary, select_protocol_from_pilot

NOTEBOOK_DIR = Path(".").resolve()
CONFIG_PATH = NOTEBOOK_DIR / "iqm_metric_test_config.toml"
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

PHASE = "pilot"  # "pilot" or "final"
DEPTH = 2
STATEVECTOR_ONLY = False
RUN_ID = None  # set to resume an existing run directory name

spec = load_phase_spec(
    CONFIG_PATH,
    phase=PHASE,
    depth=DEPTH,
    run_iqm_hardware_override=False if STATEVECTOR_ONLY else None,
)
run_id = RUN_ID or timestamp_run_id(f"{spec.phase}_depth{spec.depth}")
run_dir = build_run_dir(spec, run_id, root=PROJECT_ROOT)
run_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Run directory: {run_dir}")
print(f"Phase={spec.phase}, depth={spec.depth}, folds={spec.folds}, shots={spec.shots}, repeats={spec.repeats}")

## 2. Offline Sanity Check

In [ ]:
import numpy as np

from qbanknote.ansatzes import odra_ansatz, simulator_ansatz
from qbanknote.weights import metric_weight_path

for ansatz_name, builder in [("odra", odra_ansatz), ("simulator", simulator_ansatz)]:
    qc = builder(spec.num_qubits, spec.depth)
    weight = metric_weight_path(
        spec.depth,
        ansatz_name,
        fold=1,
        epoch=spec.checkpoint_epoch,
        simulator_uses_ideal_suffix=spec.simulator_uses_ideal_suffix,
        root=PROJECT_ROOT,
    )
    print(f"{ansatz_name}: {len(qc.parameters)} params, weight template -> {weight.name}")

labels = predictions_to_labels(np.array([0.2, -0.1, 0.0]))
assert set(labels.tolist()) == {-1.0, 1.0}
print("Offline sanity check passed.")

## 3. Run Experiment (Statevector + Optional Hardware)

Set `STATEVECTOR_ONLY = True` to skip IQM jobs. Re-running this cell resumes from CSV checkpoints in `run_dir`.

In [ ]:
iqm_backend = None
if spec.run_iqm_hardware:
    iqm_backend = connect_to_iqm_backend(spec.iqm_url)
    print(f"Connected to backend: {iqm_backend}")

run_cv_experiment(
    spec,
    run_dir,
    iqm_backend=iqm_backend,
    root=PROJECT_ROOT,
)

statevector_df = read_csv_or_empty(run_dir / "statevector_results.csv")
run_df = read_csv_or_empty(run_dir / "run_level_results.csv")
summary_df = summarize_results(spec, statevector_df=statevector_df, run_df=run_df)
summary_df.to_csv(run_dir / "summary_comparison.csv", index=False)
summary_df

## 4. Pilot Protocol Selection

Run this after a completed **pilot** phase to freeze shot count and repeat budget for the final run.

In [ ]:
pilot_spec = load_phase_spec(CONFIG_PATH, phase="pilot", depth=DEPTH)
pilot_summary = read_csv_or_empty(run_dir / "summary_comparison.csv")
protocol = select_protocol_from_pilot(pilot_summary, pilot_spec)

if not protocol["shot_stability"].empty:
    protocol["shot_stability"].to_csv(run_dir / "shot_stability.csv", index=False)
if not protocol["shot_stability_aggregate"].empty:
    protocol["shot_stability_aggregate"].to_csv(
        run_dir / "shot_stability_aggregate.csv", index=False
    )

report = {k: v for k, v in protocol.items() if not isinstance(v, pd.DataFrame)}
report_path = run_dir / "protocol_recommendation.json"
report_path.write_text(pd.Series(report).to_json(indent=2) + "\n")
report

## 5. Final Run With Frozen Protocol

Update `PHASE = "final"` and optionally override shots/repeats from `protocol_recommendation.json`.

In [ ]:
import json

protocol_path = run_dir / "protocol_recommendation.json"
if protocol_path.exists():
    recommendation = json.loads(protocol_path.read_text())
    final_shots = [int(recommendation["chosen_shot"])]
    final_repeats = int(recommendation["chosen_repeats"])
else:
    final_shots = None
    final_repeats = None

final_spec = load_phase_spec(
    CONFIG_PATH,
    phase="final",
    depth=DEPTH,
    shots_override=final_shots,
    repeats_override=final_repeats,
)
final_run_id = timestamp_run_id(f"final_depth{final_spec.depth}")
final_run_dir = build_run_dir(final_spec, final_run_id, root=PROJECT_ROOT)

final_backend = connect_to_iqm_backend(final_spec.iqm_url) if final_spec.run_iqm_hardware else None
run_cv_experiment(
    final_spec,
    final_run_dir,
    iqm_backend=final_backend,
    root=PROJECT_ROOT,
)
print(f"Final run directory: {final_run_dir}")

## 6. Statistical Analysis & Plots

In [ ]:
analysis_dir = final_run_dir if "final_run_dir" in globals() else run_dir
summary_df = read_csv_or_empty(analysis_dir / "summary_comparison.csv")
analysis = analyze_final_summary(summary_df)

for name, frame in analysis.items():
    if not frame.empty:
        frame.to_csv(analysis_dir / f"{name}.csv", index=False)

analysis["paired_tests"]

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ["iqm_mean_accuracy", "iqm_mean_f1"]):
        pivot = summary_df.pivot_table(index="fold", columns="ansatz", values=metric)
        pivot.plot(kind="bar", ax=ax)
        ax.set_title(metric)
        ax.set_xlabel("fold")
    fig.tight_layout()
    plt.show()